In [22]:
import torch # 딥러닝을 하기 위한 조리대
import torchvision # 파이토치 중 이미지를 다루는 도구함
import torch.nn as nn # 신경망 층을 만들 재료 가져오기
from torchvision.transforms import ToTensor # 텐서로 바꿔주는 변환기
torch.manual_seed(0) # 무작위 시작점을 0으로 고정

In [23]:
data_train = torchvision.datasets.MNIST('./data', download=True, train=True,
transform=ToTensor()) # traindata 가져오기

data_test  = torchvision.datasets.MNIST('./data', download=True, train=False,
transform=ToTensor()) # testdata 가져오기

In [24]:
print('Training samples:', len(data_train)) #  train data

Training samples: 60000


In [25]:
print('Test samples:', len(data_test)) # test data

Test samples: 10000


In [26]:
img, label = data_train[0] # train test 중에서 첫 번째 상자에서 image 와 label 꺼내기

In [27]:
print('Tensor shape:',img.shape) # [ 채널 수, 세로 길이, 가로 길이]
# 질문: 채널 수가 1이면 흑백 3이면 컬러인데 0이나 2면 머지?
# -> 정상적인 이미지는 불가 0 이면 아무것도 없음, 2이면 현실 이미지에 없음

Tensor shape: torch.Size([1, 28, 28])


In [28]:
print('Label:',label) # 라벨 = 정답, 겉보기 숫자
# 질문 많은 라벨이 있는데 왜 5가 나오지?
# -> 위에서 이미 [0]으로 인덱싱 했었음

Label: 5


In [29]:
print('Min/Max pixel:', img.min().item(),img.max().item())
# img.min 이미지 숫자판의 작은 숫자, img.max 이미지 숫자판 큰 숫자 .item() 순수한 파이썬 숫자
# 출력 결과의 의미 0.0  최솟값(검은색) 1.0 최댓값(흰색)
# 질문 색깔 정보들의 밝기를 어떻게 결정하는지(예를들어 빨강이랑 파랑 중에 누가 더 밝냐?)
# -> 흑백 전환 공식이 존재한다 밝기 = (빨강 x 0.299) + (초록 x 0.587) + (파랑 x 0.114)
# -> 빨강 (1,0,0) 초록 (0,1,0) 질문 주황색이런거는? -> (1,0.5,0) 이런식으로 조합을 한

Min/Max pixel: 0.0 1.0


In [30]:
print('First 10 lavels;',[data_train[i][1] for i in range(10)])


First 10 lavels; [5, 0, 4, 1, 9, 2, 1, 3, 1, 4]


In [31]:
x_train = data_train.data.numpy().reshape(60000,-1)
y_train = data_train.targets.numpy()
# -1: 전체 데이터 개수 유지, 차원의 크기 자동 계산
x_test = data_test.data.numpy().reshape(10000,-1)
y_test = data_test.targets.numpy()

In [32]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)
# (10000,) 1차원
# (10000, 1) 2차원
# (10000, 784) 2차원

(60000, 784)
(60000,)
(10000, 784)
(10000,)


In [36]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn_clf = KNeighborsClassifier()

param_grid = {
      "n_neighbors": [3, 4, 5], # 몇 명의 이웃을 볼지
      "weights": ["uniform","distance"]
} # uniform: 이웃을 모두 똑같이 봄 # distance: 가까운 이웃에게 더 큰 비중

In [37]:
gird_search = GridSearchCV(
    knn_clf, # 튜닝할 모델
    param_grid, # 시험할 하이퍼파라미터 조합
    cv=3, # 데이터를 3개로 나눠서 교차검증
    scoring="accuracy", # 정확도 기준 모델 선택
    n_jobs =-1 # 가능한 cpu 코어 모두 사용
)

In [38]:
gird_search.fit(x_train,y_train)


GridSearchCV(cv=3, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'n_neighbors': [3, 4, 5],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy')

In [42]:
grid_search = gird_search
print(grid_search.best_params_)
print(grid_search.best_score_)

{'n_neighbors': 4, 'weights': 'distance'}
0.9703500000000002


In [46]:
from sklearn.metrics import accuracy_score

y_pred = grid_search.predict(x_test)
accuracy = accuracy_score(y_test,y_pred)

print(accuracy)

0.9714
